# NB-R10 — Results Compilation and Summary Tables

**Pipeline stage:** 10 of 13

**Purpose.** Collect the outputs of NB-R01 through NB-R09 into consolidated summary tables, and produce a single tracking file cross-referencing each methodological issue addressed during pipeline development to the notebook that resolves it.

**Inputs:** all `results/*.csv` and `results/*.json` files produced by NB-R01 through NB-R09.

**Outputs:** consolidated summary tables (splits, feature specification, hyperparameters, regime metrics, statistical tests, trading simulation) and `results/issue_tracker.csv`.

**Note:** this notebook closes out the initial pipeline. Later additions (GARCH comparison, extended threshold sensitivity, 1-day/5-day model retraining, and figure regeneration) are covered by NB-R11 through NB-R13.


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

PROJ    = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC    = PROJ / 'data' / 'processed'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'

print('Loading all result files...')

# Load all results
split_summary   = pd.read_csv(PROC / 'split_summary.csv')
hp_table        = pd.read_csv(RESULTS / 'hyperparameter_table.csv')
stats           = json.load(open(RESULTS / 'statistical_tests.json'))
baselines       = pd.read_csv(RESULTS / 'baseline_comparison.csv')
ablation        = pd.read_csv(RESULTS / 'ablation_results.csv')
trading         = pd.read_csv(RESULTS / 'trading_simulation.csv')
walkforward     = pd.read_csv(RESULTS / 'walkforward_results.csv')
vix_cfg         = json.load(open(RESULTS / 'vix_threshold_config.json'))
threshold_comp  = pd.read_csv(RESULTS / 'threshold_comparison.csv')

print('[OK] All results loaded.')

## Table 1: Clean Data Splits

In [ ]:
print('TABLE 1: Clean Data Splits (label-leakage-safe split boundaries)')
print(split_summary.to_string(index=False))

## Table 2: Feature Engineering Specification

In [ ]:
feat_spec = pd.DataFrame([
    {'Feature': 'MACD',          'Formula/Params': 'EMA(12) - EMA(26); EMAs use exponential weighting, adjust=False',         'Leakage-safe': 'Yes'},
    {'Feature': 'MACD Signal',   'Formula/Params': 'EMA(9) of MACD',                                                           'Leakage-safe': 'Yes'},
    {'Feature': 'MACD Histogram','Formula/Params': 'MACD - MACD Signal',                                                        'Leakage-safe': 'Yes'},
    {'Feature': 'EMA-20',        'Formula/Params': 'Exponential MA, span=20, adjust=False',                                     'Leakage-safe': 'Yes'},
    {'Feature': 'RSI-14',        'Formula/Params': '100 - 100/(1+AvgGain/AvgLoss); Wilder smoothing (com=13)',                  'Leakage-safe': 'Yes'},
    {'Feature': 'Stoch %K',      'Formula/Params': '100*(Close-Low14)/(High14-Low14); 14-day lookback',                         'Leakage-safe': 'Yes'},
    {'Feature': 'Stoch %D',      'Formula/Params': '3-day SMA of %K',                                                           'Leakage-safe': 'Yes'},
    {'Feature': 'ROC-10',        'Formula/Params': '(Close/Close[t-10] - 1) * 100; 10-day rate of change',                      'Leakage-safe': 'Yes'},
    {'Feature': 'BB Upper',      'Formula/Params': 'SMA(20) + 2*std(20); simple MA, 20-day window',                             'Leakage-safe': 'Yes'},
    {'Feature': 'BB Lower',      'Formula/Params': 'SMA(20) - 2*std(20)',                                                        'Leakage-safe': 'Yes'},
    {'Feature': 'BB Width',      'Formula/Params': '(BB_Upper - BB_Lower) / SMA(20)',                                            'Leakage-safe': 'Yes'},
    {'Feature': 'ATR-14',        'Formula/Params': 'Wilder EMA(com=13) of True Range; TR=max(H-L, |H-C[t-1]|, |L-C[t-1]|)',    'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 1', 'Formula/Params': 'log(Close[t] / Close[t-1])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 2', 'Formula/Params': 'log(Close[t] / Close[t-2])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 3', 'Formula/Params': 'log(Close[t] / Close[t-3])',                                                 'Leakage-safe': 'Yes'},
    {'Feature': 'Log Ret Lag 5', 'Formula/Params': 'log(Close[t] / Close[t-5])',                                                 'Leakage-safe': 'Yes'},
])
print('TABLE 2: Feature Engineering Specification')
print(feat_spec.to_string(index=False))
feat_spec.to_csv(RESULTS / 'feature_specification.csv', index=False)

## Table 3: Hyperparameters

In [ ]:
print('TABLE 3: Final Hyperparameters (from Optuna, 50 trials, TimeSeriesSplit n=5)')
print(hp_table.to_string(index=False))

## Table 4: Regime-Stratified Accuracy (with Class Metrics)

In [ ]:
print('TABLE 4: Regime-Stratified Accuracy and Class-Stratified Metrics (Stacking Ensemble)')
regime_summary = {
    'Metric': ['N', 'Up% (local)', 'Majority-class baseline (%)', 'Accuracy (%)',
               'Precision (%)', 'Recall (%)', 'F1 (%)', 'ROC-AUC'],
    'High-VIX': [
        stats['high_vix']['n'],
        f"{stats['high_vix']['up_pct']:.1f}%",
        f"{stats['high_vix']['majority_baseline_acc']:.1f}%",
        f"{stats['high_vix']['accuracy']:.1f}%",
        f"{stats['high_vix']['precision']:.1f}%",
        f"{stats['high_vix']['recall']:.1f}%",
        f"{stats['high_vix']['f1']:.1f}%",
        f"{stats['high_vix']['auc']:.4f}",
    ],
    'Low-VIX': [
        stats['low_vix']['n'],
        f"{stats['low_vix']['up_pct']:.1f}%",
        f"{stats['low_vix']['majority_baseline_acc']:.1f}%",
        f"{stats['low_vix']['accuracy']:.1f}%",
        f"{stats['low_vix']['precision']:.1f}%",
        f"{stats['low_vix']['recall']:.1f}%",
        f"{stats['low_vix']['f1']:.1f}%",
        f"{stats['low_vix']['auc']:.4f}",
    ]
}
t4 = pd.DataFrame(regime_summary)
print(t4.to_string(index=False))
t4.to_csv(RESULTS / 'table4_regime_metrics.csv', index=False)

## Table 5: Statistical Tests Summary

In [ ]:
t5 = pd.DataFrame([
    {'Test': 'Accuracy difference (High - Low)',      'Value': f"{stats['accuracy_diff_pp']:.1f} pp", 'Note': ''},
    {'Test': 'Fisher exact (non-overlapping, every 21st day)', 'Value': f"{stats['fisher_non_overlap_p']:.3e}", 'Note': 'Avoids serial correlation'},
    {'Test': 'Fisher Bonferroni-corrected (x3)',      'Value': f"{stats['fisher_bonferroni_p']:.3e}", 'Note': '3 horizons tested'},
    {'Test': 'Block bootstrap 95% CI (block=21)',     'Value': f"[{stats['block_boot_ci_95'][0]:.1f}, {stats['block_boot_ci_95'][1]:.1f}] pp", 'Note': 'Circular block bootstrap'},
    {'Test': 'Block permutation p-value',             'Value': f"{stats['block_perm_p']:.3e}", 'Note': 'Block shuffle of regime labels'},
    {'Test': 'Block permutation Bonferroni (x3)',     'Value': f"{stats['block_perm_bonferroni_p']:.3e}", 'Note': ''},
])
print('TABLE 5: Statistical Tests (Adjusted for Overlapping Observations)')
print(t5.to_string(index=False))
t5.to_csv(RESULTS / 'table5_statistical_tests.csv', index=False)

## Table 8: Trading Simulation Summary

In [ ]:
print('TABLE 8: Trading Simulation Results')
print(f'Sharpe convention: (mean excess daily return / std) * sqrt({252})')
print('Risk-free rate: 6.5% p.a. (India 91-day T-bill, FY2025 avg = 6.5%)')
print()
print(trading.to_string(index=False))

# Highlight the headline finding
for _, row in trading.iterrows():
    if row['cost_bps'] == 50:
        status = 'above' if row['strat_above_bh'] else 'BELOW'
        print(f"\nAt 50bps: strategy Sharpe={row['strat_sharpe']:.3f} is {status} BH={row['bh_sharpe']:.3f}")

## Final Summary: Issues Identified and Fixes Applied

In [ ]:
summary = [
    {'Category': 'Look-ahead bias',        'Issue': 'Regime threshold computed from the test-period VIX distribution',        'Fix': 'NB-R02: threshold fixed from train+val distribution only',      'Status': '[DONE]'},
    {'Category': 'Label leakage',           'Issue': 'Boundary rows near each split edge could leak future information into labels', 'Fix': 'NB-R01: last 21 rows trimmed at each split boundary',            'Status': '[DONE]'},
    {'Category': 'Statistical validity',    'Issue': 'Overlapping 21-day-ahead labels violate independence assumptions in significance testing', 'Fix': 'NB-R04: block bootstrap + non-overlapping Fisher test', 'Status': '[DONE]'},
    {'Category': 'Statistical validity',    'Issue': 'High-VIX test days concentrated in a single episode, limiting generalizability', 'Fix': 'NB-R05: annual walk-forward validation across independent years', 'Status': '[DONE]'},
    {'Category': 'Statistical validity',    'Issue': 'Regime-specific baselines and target-realization timing needed clarification', 'Fix': 'NB-R04 + NB-R10: within-regime majority baselines reported',    'Status': '[DONE]'},
    {'Category': 'Trading simulation',      'Issue': 'Reported Sharpe ratio at 50bps transaction cost was factually incorrect',   'Fix': 'NB-R09: cost-sensitivity table and text updated',              'Status': '[DONE]'},
    {'Category': 'Trading simulation',      'Issue': 'Signal timing and execution protocol were ambiguous',                      'Fix': 'NB-R09: explicit execution protocol stated (signal/entry/exit timing)', 'Status': '[DONE]'},
    {'Category': 'Reporting completeness',  'Issue': 'Class-stratified metrics (precision/recall/F1) missing within each regime', 'Fix': 'NB-R04: per-regime precision, recall, F1 reported',              'Status': '[DONE]'},
    {'Category': 'Reporting completeness',  'Issue': 'Final hyperparameters and reference metadata were missing or incorrect',    'Fix': 'NB-R03: hyperparameter table added; reference metadata fixed', 'Status': '[DONE]'},
    {'Category': 'Reporting completeness',  'Issue': 'SHAP feature-importance claims made without showing computed values',       'Fix': 'NB-R07: regime-specific SHAP values and plots',                  'Status': '[DONE]'},
    {'Category': 'Reporting completeness',  'Issue': 'Feature-engineering specification needed for reproducibility',              'Fix': 'NB-R10: full feature specification table',                       'Status': '[DONE]'},
    {'Category': 'Model robustness',        'Issue': 'No comparison against simple baseline models under the same regime filter', 'Fix': 'NB-R06: majority-class, persistence, and logistic-regression baselines', 'Status': '[DONE]'},
    {'Category': 'Model robustness',        'Issue': 'No ablation of individual ensemble components',                            'Fix': 'NB-R08: per-model and no-attention ablation studies',            'Status': '[DONE]'},
    {'Category': 'Documentation',           'Issue': 'Literature references were outdated',                                      'Fix': 'Reference list updated with more recent sources',                'Status': '[DONE]'},
]

fix_df = pd.DataFrame(summary)
print(fix_df.to_string(index=False))
fix_df.to_csv(RESULTS / 'issue_tracker.csv', index=False)
print('\nIssue tracker saved to results/issue_tracker.csv')


---
## Summary

**Pipeline stage:** 10 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `results/issue_tracker.csv`
- `consolidated summary tables (see results/)`

**Next notebook:** `NB-R11_garch_and_threshold_extras.ipynb`
